# Ink-based Connected Components — Hybrid v2 + selective v3 recovery

**Strategy:** v2 remains the main algorithm. The optional v3-style morphological closing pass contributes only *bridge evidence* for broken/fine strokes; it does not create the final bounding boxes and does not use v3's padded bounding-box halo to merge groups.

### What this protects and improves
- **Preserves v2's separation logic:** seed groups merge based on actual ink distance plus v2's guarded containment rule.
- **Recovers fragmented strokes:** when closing connects several original components, unattached components may join a group if the closed cluster maps to exactly one seed group.
- **Avoids ambiguous bridges:** if a closed cluster touches two separate seed groups, the v3 recovery is ignored for that cluster.
- **Keeps small standalone figures:** no page-area threshold is imposed on returned groups, and v2's row-aware reading order is retained.
- **Preserves original geometry:** output boxes use original-ink components, not the dilated/closed mask.

Start with `CLOSE_PX = 5`, as in v3. To disable the recovery layer for an A/B comparison, pass `close_px=0` to `find_figures` or `ink_based_connected_components`. If fine strokes remain fragmented, tune `CLOSE_PX` modestly (e.g. 3–7); avoid a large halo or large closing kernel because either can link neighboring figures.

The notebook writes debug overlays to `../../results/figure_extractor_hybrid_test_output/`, leaving the v2/v3 output folders untouched.

## Core extractor

The first code cell defines the hybrid `find_figures()` function. The v2 public parameters stay in their original order; the new options are appended (`close_px`, `recover_micro`) so existing positional calls remain compatible.

In [1]:
"""Ink-based connected-component figure extraction (hybrid v2 + selective v3 recovery).

Design:
- V2 remains the source of truth for seed detection, ink-distance seed merging,
  and nearest-figure attachment.
- V3-style morphological closing is used only as supporting evidence that
  original components belong to one broken/fine-stroke structure.
- Closing is NOT used to define output bounding boxes, and V3's padded-box halo
  merge is deliberately not used. This protects separate figures with close or
  overlapping bounding boxes.

Input ink is uint8 with foreground ink=255. Distances are calibrated at 300 dpi.
"""
import cv2 as cv
import numpy as np

# --- V2 parameters (px @ 300 dpi) -------------------------------------------------
MIN_AREA  = 20      # Ignore isolated speckles smaller than this (unless recovered below)
SEED_AREA = 3000    # Components at least this large are figure seeds
TOUCH     = 8       # Merge seeds when their actual ink nearly touches
NEAR      = 45      # Additional seed merge distance, but only under V2's containment rule
CONTAIN   = 0.5     # Fraction of smaller seed box inside larger seed box for near-merge
ATTACH    = 48      # Attach non-seed components to the nearest figure within this distance

# --- Selective V3 recovery ---------------------------------------------------------
CLOSE_PX       = 5  # Morphological closing size (px @ 300 dpi); set 0/1 to disable
MICRO_MIN_AREA = 3  # Tiny component lower bound when supported by a valid closed stroke cluster


def area(b):
    return max(0, (b[2] - b[0])) * max(0, (b[3] - b[1]))


def frac_inside(piece, host):
    """Share of `piece`'s box inside `host`'s box (overlap / piece area)."""
    pa = area(piece)
    if pa <= 0:
        return 0.0
    iw = min(piece[2], host[2]) - max(piece[0], host[0])
    ih = min(piece[3], host[3]) - max(piece[1], host[1])
    return max(iw, 0) * max(ih, 0) / pa


def _dist_to(mask):
    """Distance of every pixel to the nearest True pixel of `mask`."""
    return cv.distanceTransform(np.where(mask, 0, 1).astype(np.uint8), cv.DIST_L2, 5)


def _union(a, b):
    return [min(a[0], b[0]), min(a[1], b[1]), max(a[2], b[2]), max(a[3], b[3])]


def _closing_membership(ink, labels, min_area, close_px):
    """Return original-component -> closed-cluster mapping and cluster members.

    Closing supplies a possible bridge between original components. The original
    labels and component boxes remain untouched, which avoids the bbox inflation
    and accidental cross-figure merges that can occur when the closed image is
    used as the primary segmentation.
    """
    if close_px <= 1:
        return {}, {}, {}

    kernel = cv.getStructuringElement(cv.MORPH_RECT, (int(close_px), int(close_px)))
    closed = cv.morphologyEx(ink, cv.MORPH_CLOSE, kernel)
    _, closed_labels, closed_stats, _ = cv.connectedComponentsWithStats(closed, connectivity=8)

    common = (labels > 0) & (closed_labels > 0)
    if not np.any(common):
        return {}, {}, {}

    pairs = np.column_stack((closed_labels[common], labels[common]))
    unique_pairs, counts = np.unique(pairs, axis=0, return_counts=True)

    # Assign each original component to the closing cluster with the largest
    # shared-ink overlap. (Normally this is a one-to-one membership.)
    best_cluster = {}
    for (cluster_id, component_id), count in zip(unique_pairs, counts):
        component_id = int(component_id)
        cluster_id = int(cluster_id)
        current = best_cluster.get(component_id)
        if current is None or count > current[1]:
            best_cluster[component_id] = (cluster_id, int(count))

    component_cluster = {i: value[0] for i, value in best_cluster.items()}
    cluster_components = {}
    for component_id, cluster_id in component_cluster.items():
        cluster_components.setdefault(cluster_id, set()).add(component_id)

    cluster_area = {
        int(cid): int(closed_stats[cid, cv.CC_STAT_AREA])
        for cid in cluster_components
    }
    return component_cluster, cluster_components, cluster_area


def find_figures(ink, min_area=MIN_AREA, seed_area=SEED_AREA, touch=TOUCH, near=NEAR,
                 contain=CONTAIN, attach=ATTACH, dpi=300, close_px=CLOSE_PX,
                 recover_micro=True):
    """Return figure boxes as [x1, y1, x2, y2] with exclusive x2/y2.

    V2's seed detection and seed-merging decisions use the *original ink*.
    V3-style closing is only allowed to reconnect small components when their
    closed cluster maps to exactly one V2 seed group. If a closing cluster spans
    multiple distinct seed groups, its bridge evidence is ignored.

    Args:
        ink: uint8 binary image, foreground ink=255.
        min_area, seed_area, touch, near, contain, attach: V2 thresholds.
        dpi: Scan resolution; pixel thresholds scale from 300 dpi.
        close_px: Morphological-closing size at 300 dpi; <=1 disables recovery.
        recover_micro: Permit components below min_area (but >= MICRO_MIN_AREA)
            when closing connects them to a valid component cluster.
    """
    if ink is None or ink.ndim != 2:
        raise ValueError("ink must be a 2-D grayscale/binary image with ink=255")

    H, W = ink.shape
    s = float(dpi) / 300.0
    if s <= 0:
        raise ValueError("dpi must be positive")

    min_area_px = min_area * s * s
    seed_area_px = seed_area * s * s
    touch_px, near_px, attach_px = touch * s, near * s, attach * s
    close_px_scaled = int(round(close_px * s))

    n, labels, stats, _ = cv.connectedComponentsWithStats(ink, connectivity=8)
    x, y, w, h, component_area = (stats[:, k].astype(np.int64) for k in range(5))
    boxes = np.stack([x, y, x + w, y + h], axis=1)
    fill = component_area / np.maximum(w * h, 1)
    aspect = np.maximum(w, h) / np.maximum(np.minimum(w, h), 1)

    big = component_area >= seed_area_px
    # Keep V2's scan-artifact guard for large components.
    junk = big & (((aspect >= 15) & (np.minimum(w, h) < 0.03 * W)) | (fill > 0.8))
    base_keep = (component_area >= min_area_px) & ~junk
    base_keep[0] = False

    component_cluster, cluster_components, cluster_area = _closing_membership(
        ink, labels, min_area_px, close_px_scaled
    )

    # Recover sub-minimum broken-stroke fragments only if their closed cluster
    # is large enough and also contains at least one ordinary, non-junk component.
    micro_keep = np.zeros(n, dtype=bool)
    if recover_micro and cluster_components:
        for cid, member_ids in cluster_components.items():
            if cluster_area.get(cid, 0) < min_area_px:
                continue
            if not any(base_keep[i] for i in member_ids if i < n):
                continue
            for i in member_ids:
                if (0 < i < n and component_area[i] >= MICRO_MIN_AREA
                        and component_area[i] < min_area_px and not junk[i]):
                    micro_keep[i] = True

    keep = base_keep | micro_keep
    keep[0] = False
    seeds = [int(i) for i in np.argsort(-component_area) if keep[i] and big[i]]
    seed_ids = set(seeds)
    pieces = {int(i) for i in range(1, n) if keep[i] and not big[i]}

    # ---- 1. V2 seed grouping: actual ink distance + guarded containment -------------
    groups = [dict(ids=[i], box=list(map(int, boxes[i])), ink=int(component_area[i]))
              for i in seeds]

    def mask_of(group, roi):
        return np.isin(labels[roi[1]:roi[3], roi[0]:roi[2]], group["ids"])

    def ink_gap(g1, g2):
        b1, b2 = g1["box"], g2["box"]
        box_gap = np.hypot(
            max(b1[0] - b2[2], b2[0] - b1[2], 0),
            max(b1[1] - b2[3], b2[1] - b1[3], 0),
        )
        if box_gap > near_px:
            return np.inf
        roi = _union(b1, b2)
        m1, m2 = mask_of(g1, roi), mask_of(g2, roi)
        if not m1.any() or not m2.any():
            return np.inf
        return float(_dist_to(m1)[m2].min())

    merged = True
    while merged:
        merged = False
        for p in range(len(groups)):
            for q in range(p + 1, len(groups)):
                host, small = sorted((groups[p], groups[q]), key=lambda g: -g["ink"])
                gap = ink_gap(host, small)
                if gap <= touch_px or (
                    gap <= near_px and frac_inside(small["box"], host["box"]) >= contain
                ):
                    host["ids"] += small["ids"]
                    host["box"] = _union(host["box"], small["box"])
                    host["ink"] += small["ink"]
                    groups.remove(small)
                    merged = True
                    break
            if merged:
                break

    # ---- 2. V2 nearest-ink attachment + conservative V3 bridge fallback -------------
    for _ in range(8):
        claims = {}  # component -> (distance, group index)
        seed_group = {
            sid: gi
            for gi, group in enumerate(groups)
            for sid in group["ids"]
            if sid in seed_ids
        }

        for gi, group in enumerate(groups):
            gx1, gy1 = max(int(group["box"][0] - attach_px), 0), max(int(group["box"][1] - attach_px), 0)
            gx2 = min(int(group["box"][2] + attach_px) + 1, W)
            gy2 = min(int(group["box"][3] + attach_px) + 1, H)
            roi_labels = labels[gy1:gy2, gx1:gx2]
            dt = None
            for i in pieces:
                bx = boxes[i]
                if bx[2] < gx1 or bx[0] > gx2 or bx[3] < gy1 or bx[1] > gy2:
                    continue
                if dt is None:
                    dt = _dist_to(np.isin(roi_labels, group["ids"]))
                ys = slice(max(int(bx[1]), gy1) - gy1, min(int(bx[3]), gy2) - gy1)
                xs = slice(max(int(bx[0]), gx1) - gx1, min(int(bx[2]), gx2) - gx1)
                own = roi_labels[ys, xs] == i
                if own.any():
                    d = float(dt[ys, xs][own].min())
                    if d <= attach_px and d < claims.get(i, (np.inf,))[0]:
                        claims[i] = (d, gi)

        # Only if ordinary V2 distance attachment did not claim a component,
        # allow closing to bridge it. A closing cluster touching multiple distinct
        # seed groups is ambiguous and is explicitly not used for recovery.
        for i in pieces:
            if i in claims:
                continue
            cid = component_cluster.get(i)
            if cid is None:
                continue
            candidate_groups = {
                seed_group[sid]
                for sid in cluster_components.get(cid, set())
                if sid in seed_group
            }
            if len(candidate_groups) == 1:
                gi = next(iter(candidate_groups))
                claims[i] = (attach_px + 1.0, gi)

        if not claims:
            break
        for i, (_, gi) in claims.items():
            groups[gi]["ids"].append(i)
            groups[gi]["box"] = _union(groups[gi]["box"], list(map(int, boxes[i])))
            groups[gi]["ink"] += int(component_area[i])
            pieces.discard(i)

    return [[int(v) for v in g["box"]] for g in groups]


def reading_order(boxes):
    """Rows top-to-bottom; boxes sharing >=50% of height form a row."""
    rows = []
    for b in sorted(boxes, key=lambda b: b[1]):
        for row in rows:
            top, bottom = min(c[1] for c in row), max(c[3] for c in row)
            if min(bottom, b[3]) - max(top, b[1]) >= 0.5 * min(bottom - top, b[3] - b[1]):
                row.append(b)
                break
        else:
            rows.append([b])
    return [b for row in rows for b in sorted(row, key=lambda b: b[0])]


## Run on the test folder

This cell uses the same input folder as v2 and writes overlays to a **new** hybrid output folder. If the notebook environment doesn't have that relative data folder, update `directory_path` before running this cell. `ink_based_connected_components()` also returns the detected boxes, which makes it easy to add evaluation or compare v2-only (`close_px=0`) against hybrid results.

In [2]:
import glob
import os
from pathlib import Path as PATH


def ink_based_connected_components(src, debug_out, dpi=300, close_px=CLOSE_PX):
    gray = cv.imread(src, cv.IMREAD_GRAYSCALE)
    if gray is None:
        raise FileNotFoundError(f"Could not read input image: {src}")
    _, ink = cv.threshold(gray, 128, 255, cv.THRESH_BINARY_INV)
    figures = reading_order(find_figures(ink, dpi=dpi, close_px=close_px))

    debug = cv.cvtColor(ink, cv.COLOR_GRAY2BGR)
    for i, (x1, y1, x2, y2) in enumerate(figures, 1):
        cv.rectangle(debug, (x1, y1), (x2 - 1, y2 - 1), (0, 255, 0), 2)
        cv.putText(debug, str(i), (x1 + 8, y1 + 40), cv.FONT_HERSHEY_SIMPLEX, 1.4, (0, 255, 0), 3)
        # To save each crop separately, uncomment:
        # cv.imwrite(f"figure_{i}.png", gray[y1:y2, x1:x2])

    out_path = PATH(debug_out)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if not cv.imwrite(str(out_path), debug):
        raise IOError(f"Could not write debug image: {out_path}")
    return figures


# Same test data location as v2; output goes to a new folder to avoid overwriting.
directory_path = "../../data/figure_extractor_test/"
all_sketch_pages = glob.glob(directory_path + "*.png")
debug_directory = PATH("../../results/figure_extractor_hybrid_test_output/")
debug_directory.mkdir(parents=True, exist_ok=True)

for src in all_sketch_pages:
    debug_out = debug_directory / f"debug_{PATH(src).name}"
    figures = ink_based_connected_components(src, debug_out)
    print(f"{PATH(src).name}: {len(figures)} figures -> {debug_out}")


2393_2398_page_3.png: 3 figures -> ../../results/figure_extractor_hybrid_test_output/debug_2393_2398_page_3.png
2424_2429_page_2.png: 2 figures -> ../../results/figure_extractor_hybrid_test_output/debug_2424_2429_page_2.png
2424_2429_page_3.png: 2 figures -> ../../results/figure_extractor_hybrid_test_output/debug_2424_2429_page_3.png
2398_2403_page_3.png: 3 figures -> ../../results/figure_extractor_hybrid_test_output/debug_2398_2403_page_3.png
sheet_pages_0003_0005.png: 8 figures -> ../../results/figure_extractor_hybrid_test_output/debug_sheet_pages_0003_0005.png
2398_2403_page_4.png: 3 figures -> ../../results/figure_extractor_hybrid_test_output/debug_2398_2403_page_4.png
sheet_pages_0010_0012.png: 14 figures -> ../../results/figure_extractor_hybrid_test_output/debug_sheet_pages_0010_0012.png
sheet_pages_0008_0009.png: 5 figures -> ../../results/figure_extractor_hybrid_test_output/debug_sheet_pages_0008_0009.png
2453_2458_page_3.png: 7 figures -> ../../results/figure_extractor_hybrid_